# Lane Line Detection with OpenCV

A classical computer-vision pipeline that detects and overlays road lane lines on video footage — no deep learning required.

### Pipeline overview

1. **Grayscale + blur** – reduce noise before edge detection
2. **Canny edge detection** – find edges
3. **Region of interest (ROI) mask** – keep only the road area (triangle/trapezoid in front of the camera)
4. **Hough Transform** – detect straight lines from edges
5. **Average/extrapolate** – merge left and right lane line segments into two clean lines
6. **Overlay** – draw the lines back onto the original frame

> See the project `README.md` for setup instructions and tuning tips.

In [28]:
import cv2
import numpy as np

#### 1. Define the region of interest 

In [29]:
def region_of_interest(img, vertices):
    mask = np.zeros_like(img)
    cv2.fillPoly(mask, vertices, 255)
    return cv2.bitwise_and(img, mask)

#### 2. Make the line points

In [30]:
def make_line_points(y1, y2, line_params):
    if line_params is None:
        return None
    slope, intercept = line_params
    if slope == 0:
        slope = 0.1
    # Get x1 and x2 from the original equation x = (y - intercept) / slope
    x1 = int((y1 - intercept) / slope)
    x2 = int((y2 - intercept) / slope)
    return [[x1, y1, x2, y2]]


#### 3. Average slope intercept

In [31]:
def average_slope_intercept(lines, img_shape):
    left_fit, right_fit = [], []
    for line in lines:
        x1, y1, x2, y2 = line.flatten() if line.ndim > 1 else line
        if x2 == x1:
            continue
        slope, intercept = np.polyfit((x1, x2), (y1, y2), 1)
        if slope < 0:
            left_fit.append((slope, intercept))
        else:
            right_fit.append((slope, intercept))

    y1 = img_shape[0]
    y2 = int(y1 * 0.7)

    left_line = make_line_points(y1, y2, np.mean(left_fit, axis=0) if left_fit else None)
    right_line = make_line_points(y1, y2, np.mean(right_fit, axis=0) if right_fit else None)
    return left_line, right_line

#### 4. Process frame

In [ ]:
def process_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 50, 150)

    height, width = frame.shape[:2]
    roi_vertices = np.array([[
        (int(0.05*width), height),
        (int(0.45*width), int(0.6*height)),
        (int(0.55*width), int(0.6*height)),
        (int(0.95*width), height)
    ]], dtype=np.int32)
    cropped = region_of_interest(edges, roi_vertices)

    lines = cv2.HoughLinesP(
        cropped, rho=2, theta=np.pi/180, threshold=120,
        minLineLength=50, maxLineGap=100
    )

    line_img = np.zeros_like(frame)
    if lines is not None:
        left, right = average_slope_intercept(lines, frame.shape)
        for line in [left, right]:
            if line is not None:
                x1, y1, x2, y2 = line[0]
                cv2.line(line_img, (x1, y1), (x2, y2), (0, 255, 0), 8)

    return cv2.addWeighted(frame, 0.8, line_img, 1, 0)

#### 5. Run

In [ ]:
# Path to the input driving video and where to save the annotated output.
# Replace VIDEO_PATH with the path to your own footage before running.
VIDEO_PATH = "sample_videos/road2.mp4"
OUTPUT_PATH = "output/output.mp4"

import os
os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(
        f"Could not find '{VIDEO_PATH}'. Update VIDEO_PATH to point to a valid video file."
    )

cap = cv2.VideoCapture(VIDEO_PATH)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, 20, (int(cap.get(3)), int(cap.get(4))))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame is None or frame.size == 0:
        break
    result = process_frame(frame)

    cv2.namedWindow("Lane Detection", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Lane Detection", 800, 600)
    cv2.imshow("Lane Detection", result)

    out.write(result)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Done. Annotated video saved to '{OUTPUT_PATH}'.")

Done. Annotated video saved to 'output/output.mp4'.
